# Meta competitivo de Dota 2: EDA y calidad de datos (2016–2026)

**Curso:** Big Data — Universidad Nacional de Juliaca (UNAJ) · 2026-2
**Docente:** Leibnihtz Abrahan Ayamamani Choque
**Entrega:** Proyecto Unidad 1 — EDA hasta calidad de datos

> **Cómo usar este notebook.** Las celdas marcadas con ✏️ son tuyas: reemplaza el texto guía por tu propio análisis.
> Las celdas con `# TODO` tienen código incompleto que debes terminar. Antes de entregar ejecuta **Kernel → Restart & Run All**.

| # | Sección | Criterio de la rúbrica |
|---|---|---|
| 0 | Preparación del entorno | Reproducibilidad (criterio 2) |
| 1 | Proyecto, tema e integrantes | 1. Tema, propósito y pregunta |
| 2 | Pregunta motivadora y utilidad | 1. Tema, propósito y pregunta |
| 3 | Fuente, período y condiciones de uso | 2. Fuente y comprensión |
| 4 | Unidad de análisis | 2. Fuente y comprensión |
| 5 | Carga reproducible y evidencia básica | 2 y 3 |
| 6 | Perfil descriptivo | 3. Descripción y diccionario |
| 7 | Diccionario de datos | 3. Descripción y diccionario |
| 8 | Evaluación de calidad | 4. Calidad y diagnóstico |
| 9 | Diagnóstico final | 4. Calidad y diagnóstico |
| 10 | Siguiente etapa (opcional) | Visión del proyecto |

## Verificar Entorno Virtual Activo

In [1]:
import sys
import pandas as pd 

print(sys.version) 
print(sys.executable)

print(pd.__version__) 
print("Entorno BigData listo") 

3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
d:\Cursos\BigData\EDA\Dota2env20262\Scripts\python.exe
3.0.6
Entorno BigData listo


---
## 0. Preparación del entorno

Esta celda localiza la raíz del proyecto (buscando el archivo `proyecto.marker`) y desde ahí construye todas las rutas.
**No hay ninguna ruta absoluta en este notebook**: funciona igual en tu PC, en la del docente o en otra carpeta.

In [2]:
import sys
from pathlib import Path

# Añadimos la raíz del proyecto al path para poder importar src/
_aqui = Path.cwd().resolve()
_raiz = next((p for p in [_aqui, *_aqui.parents] if (p / "proyecto.marker").is_file()), None)
if _raiz is None:
    raise FileNotFoundError("Abre este notebook desde dentro de la carpeta del proyecto.")
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config, ingesta
from src.calidad import EvaluadorCalidad

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

print("Raíz del proyecto :", config.RAIZ)
print("Python            :", platform.python_version())
print("pandas            :", pd.__version__)
print("Bronze            :", config.BRONZE.relative_to(config.RAIZ))

Raíz del proyecto : D:\Cursos\BigData\EDA\Proyecto_Dota2_Meta
Python            : 3.12.10
pandas            : 3.0.6
Bronze            : data\bronze


---
## 1. Proyecto, tema e integrantes ✏️

- **Nombre del proyecto:** ✏️ (ej.: *"¿Quién manda en el meta? Diversidad y equilibrio del Dota 2 profesional"*)
- **Tema elegido:** ✏️ en 2–3 líneas: qué es Dota 2 profesional y por qué es un buen caso de Big Data.
- **Integrantes:** ✏️
-  Apaza Mamani Ruben
-  Apaza Suca Elwin Junior
-  Mamani Quilca Richard
-  Quiza Apaza Kevin Manuel
-  Vilca Ochoa Nathaly Fiorella 

---
## 2. Pregunta motivadora y utilidad esperada ✏️

Plantilla de la rúbrica:
> *¿Qué aspecto del Dota 2 profesional queremos describir y cómo podría esa información ayudar a tomar una decisión, mejorar un proceso o generar valor económico o social?*

**Mi pregunta:** ✏️ ¿Qué tan equilibrado y diverso ha sido el meta competitivo de Dota 2 parche a parche, y cómo podría esa información ayudar a los analistas de equipos a priorizar su preparación de draft?

**Tipo de pregunta**: detectar una oportunidad de ahorro, ingreso o impacto

**¿Para qué serviría responderla? ¿Quién la usaría?** ✏️ Nombra al menos un usuario concreto (analista de un equipo, organizador de torneos, casa de transmisión...) y la decisión que tomaría.
-- Predecir la victoria de un equipo unicamente analizando el draft
-- Coachs de distintos equipos profesionales
-- Usuarios que apuestan en partidas profesionales

**Sub-preguntas que guiarán el EDA** (3 a 5, todas *descriptivas*, sin causalidad):
1. ✏️ ¿Cómo ha cambiado la duración de las partidas entre años o parches?
2. ✏️ ¿Qué tan concentrados están los picks en pocos héroes?
3. ✏️ ¿Qué combinaciones de heroes sugieren un mejor winrate?

> ⚠️ Esta entrega **no** exige responder la pregunta: exige dejar claro por qué vale la pena estudiarla
> y si los datos tienen la calidad para hacerlo.

---
## 3. Fuente, período y condiciones de uso

### 3.1 Ficha de la fuente ✏️

| Campo | Valor |
|---|---|
| Nombre | Dota 2 Pro League Matches 2016–2026 |
| Publicado en | Kaggle — ver `config.KAGGLE_URL` |
| Autor / mantenedor | BwandoWando  |
| Origen de los datos | OpenDota API |
| Licencia / condiciones de uso | CC0: Public Domain |
| Frecuencia de actualización | Weekly (Updated a month ago) |
| Tamaño total | ~48.5 GB (todos los años) |
| Fecha de descarga | se registra automáticamente en el manifiesto (3.3) |

### 3.2 Alcance: qué se descarga y por qué ✏️

El dataset completo no es reproducible en una laptop (48.5 GB). Se aplica la estrategia
**"ancho en el tiempo, profundo en lo reciente"**, definida en `src/config.py`:

- `main_metadata.csv` (1 fila por partida, liviano) → **todos los períodos** para ver 10 años.
- `players`, `picks_bans`, `objectives`, `teams` (pesados) → **solo el período reciente**.
- Archivos **excluidos** y su justificación → tabla de abajo.

✏️ Explica con tus palabras por qué este recorte es suficiente para tu pregunta.

In [2]:
print("Períodos con metadata :", config.PERIODOS_METADATA[0], "→", config.PERIODOS_METADATA[-1],
      f"({len(config.PERIODOS_METADATA)} carpetas)")
print("Períodos con detalle  :", config.PERIODOS_DETALLE[0], "→", config.PERIODOS_DETALLE[-1],
      f"({len(config.PERIODOS_DETALLE)} carpetas)")
print("Archivos de detalle   :", config.ARCHIVOS_DETALLE)

pd.DataFrame(config.ARCHIVOS_EXCLUIDOS.items(), columns=["archivo_excluido", "justificacion"])

Períodos con metadata : 2016 → 202608 (18 carpetas)
Períodos con detalle  : 202601 → 202608 (8 carpetas)
Archivos de detalle   : ['players.csv', 'picks_bans.csv', 'objectives.csv', 'teams.csv']


,archivo_excluido,justificacion
0,chat.csv,Texto libre de jugadores (privacidad); no apor...
1,all_word_counts.csv,Derivado del chat; mismo motivo
2,cosmetics.csv,Objetos cosméticos; sin relación con la pregunta
3,teamfights.csv,Detalle de peleas; se deja para la siguiente e...
4,radiant_gold_adv.csv,Series de ventaja de oro; siguiente etapa
5,radiant_exp_adv.csv,Series de ventaja de experiencia; siguiente etapa


### 3.3 Descarga reproducible

Requiere el token de Kaggle (`kaggle.json`), igual que en el laboratorio de la semana 6.
Si un archivo ya existe en `data/bronze/`, **no se vuelve a descargar**, así que puedes re-ejecutar sin miedo.

> 💡 Empieza probando con **un solo período** (por ejemplo `["202608"]`) para verificar que todo funciona
> antes de lanzar la descarga completa.

In [5]:
   # PRUEBA: un solo archivo liviano (borrar esta celda después)
   ingesta.descargar_periodos(["202608"], ["main_metadata.csv"])

[  1/1] 202608/main_metadata.csv ... OK (0.2 MB)


[WindowsPath('D:/Cursos/BigData/EDA/Proyecto_Dota2_Meta/data/bronze/202608/main_metadata.csv')]

In [ ]:
# Paso 1: metadata de todos los períodos (liviano)
ingesta.descargar_periodos(config.PERIODOS_METADATA, config.ARCHIVOS_METADATA)

In [ ]:
# Paso 2: tablas de detalle solo del período reciente (pesado: puede tardar)
ingesta.descargar_periodos(config.PERIODOS_DETALLE, config.ARCHIVOS_DETALLE)

In [ ]:
# Paso 3: manifiesto de trazabilidad (qué archivos, cuánto pesan, huella md5, fecha)
manifiesto = ingesta.generar_manifiesto()
print(f"Archivos en Bronze: {len(manifiesto)}  |  Tamaño total: {manifiesto['tamano_mb'].sum():,.1f} MB")
manifiesto

---
## 4. Unidad de análisis ✏️

Antes de cargar todo, mira las columnas reales de cada archivo (lectura solo del encabezado, instantánea).

In [ ]:
periodo_muestra = config.PERIODOS_DETALLE[-1]
for archivo in config.ARCHIVOS_METADATA + config.ARCHIVOS_DETALLE:
    try:
        cols = ingesta.columnas_de(archivo, periodo_muestra)
        print(f"\n{archivo}  ({len(cols)} columnas)\n  ", ", ".join(cols[:40]), "..." if len(cols) > 40 else "")
    except FileNotFoundError:
        print(f"\n{archivo}: no descargado")

✏️ Completa con precisión **qué representa una fila** en cada tabla y cuál es su **clave**:

| Tabla | 1 fila representa... | Clave (debe ser única) | Se relaciona con |
|---|---|---|---|
| `main_metadata` | ✏️ una partida profesional | ✏️ `match_id` | — |
| `players` | ✏️ | ✏️ (`match_id`, `player_slot`)? | `main_metadata` por `match_id` |
| `picks_bans` | ✏️ | ✏️ | ✏️ |
| `objectives` | ✏️ | ✏️ | ✏️ |
| `teams` | ✏️ | ✏️ | ✏️ |

**Unidad de análisis principal de este trabajo:** ✏️ (y por qué es la adecuada para tu pregunta).

---
## 5. Carga reproducible y evidencia básica

Cargamos las tablas desde Bronze. `cargar_tabla` une todos los períodos y agrega `periodo_origen`.

> 💡 Si `players` no entra en memoria, usa `usecols=[...]` con solo las columnas de tu diccionario,
> o empieza con menos períodos. Documenta esa decisión.

In [ ]:
import time

t0 = time.perf_counter()
meta = ingesta.cargar_tabla("main_metadata.csv", config.PERIODOS_METADATA)
print(f"main_metadata cargado en {time.perf_counter() - t0:,.1f} s")

# TODO: si players es muy pesado, define aquí las columnas que realmente usarás
COLS_PLAYERS = None   # ej.: ["match_id", "player_slot", "hero_id", "kills", "deaths", "assists", "gold_per_min"]

t0 = time.perf_counter()
players = ingesta.cargar_tabla("players.csv", config.PERIODOS_DETALLE, usecols=COLS_PLAYERS)
picks = ingesta.cargar_tabla("picks_bans.csv", config.PERIODOS_DETALLE)
print(f"players + picks_bans cargados en {time.perf_counter() - t0:,.1f} s")

In [ ]:
def memoria_mb(df):
    return df.memory_usage(deep=True).sum() / 1024**2

pd.DataFrame({
    "tabla": ["main_metadata", "players", "picks_bans"],
    "filas": [len(meta), len(players), len(picks)],
    "columnas": [meta.shape[1], players.shape[1], picks.shape[1]],
    "memoria_MB": [memoria_mb(meta), memoria_mb(players), memoria_mb(picks)],
})

In [ ]:
meta.head()

In [ ]:
meta.info(memory_usage="deep")

In [ ]:
# TODO: repite head() e info() para players y picks_bans (puedes usar players.info(verbose=False) si tiene muchas columnas)

✏️ **Interpretación de la carga** (3–5 líneas): ¿cuántas partidas y jugadores hay? ¿10 filas en `players` por cada partida?
¿Qué tipos de dato llegaron mal (fechas como números, booleanos como texto, ids como float por culpa de nulos)?

---
## 6. Perfil descriptivo

### 6.1 Variables derivadas mínimas
Algunas columnas no se pueden interpretar crudas (por ejemplo, la fecha suele venir como *Unix timestamp* y la duración en segundos).

In [ ]:
# TODO: verifica en la sección 4 los nombres reales y ajusta si es necesario
if "start_time" in meta.columns:
    meta["fecha"] = pd.to_datetime(meta["start_time"], unit="s", errors="coerce")
    meta["anio"] = meta["fecha"].dt.year
if "duration" in meta.columns:
    meta["duracion_min"] = meta["duration"] / 60

meta[[c for c in ["match_id", "fecha", "anio", "duracion_min", "radiant_win", "patch"] if c in meta.columns]].head()

### 6.2 `describe()` de variables numéricas

In [ ]:
# TODO: elige SOLO las columnas numéricas relevantes para tu pregunta (no todas)
cols_num_meta = [c for c in ["duracion_min", "radiant_score", "dire_score", "first_blood_time"] if c in meta.columns]
meta[cols_num_meta].describe(percentiles=[.01, .25, .5, .75, .99]).T

### 6.3 `describe()` de variables categóricas

In [ ]:
# TODO: ajusta la lista a tus columnas categóricas
cols_cat_meta = [c for c in ["radiant_win", "patch", "leagueid", "region", "game_mode", "lobby_type"] if c in meta.columns]
meta[cols_cat_meta].astype("string").describe().T

In [ ]:
# Volumen por año: ¿está balanceado el período? (un período con pocas partidas pesa poco en las conclusiones)
if "anio" in meta.columns:
    ax = meta["anio"].value_counts().sort_index().plot(kind="bar", figsize=(10, 4), color="#2E86AB")
    ax.set(title="Partidas profesionales por año", xlabel="Año", ylabel="Partidas")
    plt.tight_layout()
    plt.savefig(config.FIGURAS / "partidas_por_anio.png", dpi=150)
    plt.show()

In [ ]:
# TODO: un perfil equivalente para players (kills, deaths, assists, gold_per_min...) y para picks_bans

✏️ **Interpretación del perfil** — no repitas la tabla; di lo que *significa*:
- ¿La duración tiene valores imposibles o sospechosos (mín. de 1–3 min, máx. de más de 2 h)?
- ¿Qué tan grande es la diferencia entre media y mediana? ¿Qué te dice sobre la forma de la distribución?
- ¿Hay años o parches con muy pocas partidas?
- ¿Qué hallazgo te sorprendió?

---
## 7. Diccionario de datos

### 7.1 Catálogo técnico (automático)

In [ ]:
def catalogo_tecnico(df, tabla):
    return pd.DataFrame({
        "tabla": tabla,
        "columna": df.columns,
        "tipo_pandas": [str(t) for t in df.dtypes],
        "nulos_pct": (df.isna().mean() * 100).round(2).values,
        "unicos": [df[c].nunique(dropna=True) for c in df.columns],
        "ejemplo": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })

catalogo = pd.concat([catalogo_tecnico(meta, "main_metadata"),
                      catalogo_tecnico(picks, "picks_bans")], ignore_index=True)
catalogo

### 7.2 Diccionario de negocio ✏️

Solo las columnas **relevantes para tu pregunta**. Formato de la rúbrica.
Puedes mantenerlo también en `docs/diccionario_datos.md`.

In [ ]:
# TODO: completa una fila por columna relevante (8–15 columnas en total)
diccionario = pd.DataFrame([
    {"tabla": "main_metadata", "columna": "match_id", "descripcion": "Identificador de la partida",
     "tipo_escala": "Identificador", "unidad_dominio": "Entero positivo, único",
     "importancia": "Clave para unir todas las tablas"},
    {"tabla": "main_metadata", "columna": "duration", "descripcion": "✏️",
     "tipo_escala": "Numérica continua", "unidad_dominio": "Segundos, > 0",
     "importancia": "✏️"},
    # ✏️ agrega más filas...
])
diccionario

---
## 8. Evaluación de calidad

Cada regla se escribe como una condición que **debe cumplirse**. El evaluador cuenta cuántos registros la incumplen.
Para cada regla debes justificar: **por qué** existe (conocimiento del dominio) y **qué distorsiona** si falla.

> Meta: al menos **2 reglas por dimensión** y al menos **2 reglas entre tablas** (integridad referencial / consistencia).
> Esto es lo que diferencia un trabajo de 1 punto de uno de 0.5.

In [ ]:
cal = EvaluadorCalidad()

### 8.1 Completitud — ¿están los datos que necesito?

In [ ]:
# EJEMPLO COMPLETO (úsalo como modelo para las demás reglas)
cal.evaluar(
    regla="patch no nulo",
    dimension="Completitud",
    tabla="main_metadata",
    cumple=meta["patch"].notna() if "patch" in meta.columns else pd.Series(False, index=meta.index),
    impacto="Sin parche no se puede comparar el meta entre parches: esas partidas quedan fuera del análisis central",
    responsable="Mantenedor del dataset / equipo (imputar por fecha)",
)

# TODO: otra regla de completitud. Ideas: equipos (radiant_team_id / dire_team_id) no nulos,
#       hero_id no nulo en players, account_id no nulo (¿jugadores anónimos?)

### 8.2 Unicidad — ¿cuento dos veces la misma cosa?

In [ ]:
# TODO: match_id único en main_metadata
# cal.evaluar("match_id único", "Unicidad", "main_metadata",
#             cumple=~meta["match_id"].duplicated(keep=False), impacto="...", responsable="...")

# TODO: (match_id, player_slot) único en players
# Pista: players.duplicated(subset=[...], keep=False)

### 8.3 Validez — ¿los valores son posibles?

Usa conocimiento del juego: una partida profesional no dura 30 segundos, las kills no son negativas,
un `hero_id` debe existir en el catálogo de héroes.

In [ ]:
# TODO: duración en un rango razonable. ¿Qué umbral usarías? Justifícalo (¿remakes? ¿abandonos?)
# cal.evaluar("duración entre X y Y min", "Validez", "main_metadata",
#             cumple=meta["duracion_min"].between(X, Y), impacto="...", responsable="...")

# TODO: kills, deaths, assists >= 0 en players

# TODO: fecha de la partida dentro del rango del dataset (2016–2026)

### 8.4 Consistencia — ¿los datos concuerdan entre sí?

Aquí está el valor diferencial: reglas que **cruzan tablas** o comparan columnas relacionadas.

In [ ]:
# TODO 1 — Integridad referencial: todo match_id de players existe en main_metadata
# Pista: players["match_id"].isin(meta["match_id"])

# TODO 2 — 10 jugadores por partida
# Pista: conteo = players.groupby("match_id").size(); luego mapea ese conteo a cada partida de meta
#        cumple = meta["match_id"].map(conteo).eq(10)   (ojo: solo para partidas del período de detalle)

# TODO 3 — Marcador consistente: radiant_score == suma de kills de los jugadores de Radiant
# Pista: los jugadores de Radiant suelen tener player_slot < 128 (verifica con tus datos)

# TODO 4 — Consistencia temporal: el año de la fecha coincide con la carpeta de origen (periodo_origen)

### 8.5 Resultado consolidado

In [ ]:
resultado = cal.resumen()
cal.guardar(config.OUTPUTS / "resultado_calidad.csv")
resultado

In [ ]:
# Visualización del cumplimiento por regla (de peor a mejor)
if not resultado.empty:
    fig, ax = plt.subplots(figsize=(10, 0.5 * len(resultado) + 1.5))
    sns.barplot(data=resultado, y="regla", x="cumplimiento_pct", hue="dimension", dodge=False, ax=ax)
    ax.set(xlim=(min(90, resultado["cumplimiento_pct"].min() - 1), 100.5),
           xlabel="% de registros que cumplen", ylabel="", title="Cumplimiento de reglas de calidad")
    plt.tight_layout()
    plt.savefig(config.FIGURAS / "cumplimiento_calidad.png", dpi=150)
    plt.show()

In [ ]:
# Inspecciona ejemplos concretos de registros que fallan (evidencia para el diagnóstico)
# TODO: cambia el nombre de la regla
# meta.loc[cal.indices_que_fallan("patch no nulo")].head()

---
## 9. Diagnóstico final ✏️

Elige los **2 o 3 problemas más importantes** (no necesariamente los de menor %, sino los que más afectan a *tu pregunta*).

| # | Problema de calidad | Dimensión | Registros / % afectado | Cómo altera la descripción o decisión | Acción propuesta | Quién debería atenderla |
|---|---|---|---|---|---|---|
| 1 | ✏️ | ✏️ | ✏️ | ✏️ | ✏️ | ✏️ |
| 2 | ✏️ | ✏️ | ✏️ | ✏️ | ✏️ | ✏️ |
| 3 | ✏️ | ✏️ | ✏️ | ✏️ | ✏️ | ✏️ |

**Conclusión (3–5 líneas):** ✏️ ¿La calidad de los datos permite confiar en el análisis para responder la pregunta de la sección 2?
¿Para qué parte sí, para cuál no y bajo qué condiciones (filtros, exclusiones)?

> Recuerda: no elimines datos sin justificación. Prefiere **marcar** registros con problemas (se hará en Silver).

---
## 10. Siguiente etapa (opcional) ✏️

Lo que **no** se exige en esta entrega, pero muestra visión:

- **Silver:** tablas tipadas en Parquet, con una columna `flag_calidad` por cada regla (en vez de borrar filas).
- **Gold:** ✏️ tablas agregadas que responderían la pregunta (ej.: estadísticas por héroe y parche, duración por parche).
- **Análisis:** ✏️ 1–2 hipótesis que se probarán en la siguiente unidad.

---
### ✅ Checklist antes de entregar
- [ ] Kernel → Restart & Run All termina sin errores
- [ ] Ninguna ruta absoluta (`D:\...`, `C:\...`)
- [ ] Todas las celdas ✏️ reemplazadas por texto propio
- [ ] Cada tabla o gráfico va seguido de una interpretación
- [ ] `outputs/manifiesto_bronze.csv` y `outputs/resultado_calidad.csv` generados